# Study 1: one-item Colab T4 backend contract

This notebook is a **standalone infrastructure sense check**, not a research experiment. It clones one exact Git commit and calls the versioned `gi_vqa.contract` runner. The selected item is fixed, contract-only, absent from the official test split, and reserved from future research splits.

The run hard-checks the Colab reference environment, immutable repository/config/data/model identities, isolation of the known built-in ms-swift 3.7.0 token-boundary defect, exact direct Transformers ↔ project training-template equivalence, deterministic generation, generated-token ↔ teacher-forced score parity, two answer-conditioned attribution maps, GPU memory, and post-attribution model stability.

Before running:

1. In Colab open **Runtime → Change runtime type**, choose **Runtime Version `2025.07`**, and select a **T4 GPU**. The default `2026.04` runtime uses Python 3.12 and is intentionally rejected by this reference contract.
2. Accept the gated `google/paligemma-3b-pt-224` licence on Hugging Face.
3. Add a Colab secret named `HF_TOKEN` and grant this notebook access to it. A read token is sufficient.
4. Commit and push the notebook, contract runner, configuration, and backend together. Paste that full 40-character commit SHA below.

Do not use a PASS from this notebook as a paper result. A PASS only authorises the next engineering gate: the restart-safe 20-item smoke run after the grouped split manifest exists.

In [ ]:
from pathlib import Path
import re

REPOSITORY_URL = "https://github.com/dizza01/VLM.git"
REPOSITORY_COMMIT = "PASTE_FULL_40_CHARACTER_COMMIT_SHA_HERE"
CHECKOUT_ROOT = Path("/content/vlm-contract")
INSTALL_SOURCE = Path("/content/gi-vqa-install-source")
ARTIFACT_DIR = Path("/content/gi_vqa_contract_artifacts")
DOWNLOAD_BUNDLE = True

if re.fullmatch(r"[0-9a-fA-F]{40}", REPOSITORY_COMMIT) is None:
    raise ValueError(
        "REPOSITORY_COMMIT must be the full 40-character SHA of the pushed commit to test"
    )
REPOSITORY_COMMIT = REPOSITORY_COMMIT.lower()
PROJECT_ROOT = CHECKOUT_ROOT / "gi_vqa_research"
CONFIG_PATH = PROJECT_ROOT / "configs" / "study1" / "smoke.yaml"
print(f"Contract commit: {REPOSITORY_COMMIT}")

In [ ]:
import hashlib
import json
import platform
import shutil
import subprocess
import sys
import traceback

def run_command(command, *, cwd=None, capture=False):
    return subprocess.run(
        [str(part) for part in command],
        cwd=cwd,
        check=True,
        text=True,
        capture_output=capture,
    )

if CHECKOUT_ROOT.exists():
    shutil.rmtree(CHECKOUT_ROOT)
if INSTALL_SOURCE.exists():
    shutil.rmtree(INSTALL_SOURCE)
if ARTIFACT_DIR.exists():
    shutil.rmtree(ARTIFACT_DIR)
ARTIFACT_DIR.mkdir(parents=True)
BOOTSTRAP_OK = False

try:
    if sys.version_info[:2] != (3, 11):
        raise RuntimeError(
            "Python 3.11 is required. In Colab select Runtime → Change runtime type → "
            f"Runtime Version 2025.07, reconnect, and run from the top. Observed: {sys.version}"
        )

    gpu_query = run_command(
        [
            "nvidia-smi",
            "--query-gpu=name,driver_version,memory.total",
            "--format=csv,noheader,nounits",
        ],
        capture=True,
    ).stdout.strip().splitlines()
    if len(gpu_query) != 1 or "T4" not in gpu_query[0]:
        raise RuntimeError(f"Exactly one T4 GPU is required; observed {gpu_query}")

    run_command(
        ["git", "clone", "--filter=blob:none", "--no-checkout", REPOSITORY_URL, CHECKOUT_ROOT]
    )
    run_command(["git", "checkout", "--detach", REPOSITORY_COMMIT], cwd=CHECKOUT_ROOT)
    resolved_commit = run_command(
        ["git", "rev-parse", "HEAD"], cwd=CHECKOUT_ROOT, capture=True
    ).stdout.strip().lower()
    if resolved_commit != REPOSITORY_COMMIT:
        raise RuntimeError(
            f"Resolved commit {resolved_commit} does not match {REPOSITORY_COMMIT}"
        )
    if not PROJECT_ROOT.is_dir() or not CONFIG_PATH.is_file():
        raise RuntimeError("The checked-out commit does not contain the expected project/configuration")

    # Install from a disposable copy so build metadata cannot dirty the verified checkout.
    shutil.copytree(PROJECT_ROOT, INSTALL_SOURCE)
    run_command(
        [sys.executable, "-m", "pip", "install", "--no-cache-dir", f"{INSTALL_SOURCE}[gpu]"]
    )
    run_command(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--force-reinstall",
            "--no-deps",
            str(INSTALL_SOURCE),
        ]
    )
    pip_check = subprocess.run(
        [sys.executable, "-m", "pip", "check"],
        check=False,
        capture_output=True,
        text=True,
    )
    pip_check_text = "\n".join(
        part.strip() for part in (pip_check.stdout, pip_check.stderr) if part.strip()
    )
    (ARTIFACT_DIR / "pip_check.txt").write_text(
        (pip_check_text + "\n") if pip_check_text else "No broken requirements found.\n",
        encoding="utf-8",
    )
    scoped_distributions = {
        "accelerate",
        "bitsandbytes",
        "datasets",
        "fsspec",
        "gi-vqa-research",
        "huggingface-hub",
        "ms-swift",
        "numpy",
        "pandas",
        "peft",
        "pillow",
        "pyyaml",
        "sentencepiece",
        "torch",
        "transformers",
        "wandb",
    }
    scoped_pip_conflicts = [
        line
        for line in pip_check_text.splitlines()
        if line and line.split(maxsplit=1)[0].lower().replace("_", "-") in scoped_distributions
    ]
    if scoped_pip_conflicts:
        raise RuntimeError(
            "Project-stack dependency conflicts: " + " | ".join(scoped_pip_conflicts)
        )
    if pip_check.returncode != 0:
        print("Global pip check WARNING (unrelated Colab packages):")
        print(pip_check_text)
    installed_contract_path = Path(
        run_command(
            [
                sys.executable,
                "-c",
                "import gi_vqa.contract; print(gi_vqa.contract.__file__)",
            ],
            capture=True,
        ).stdout.strip()
    )
    source_contract_path = PROJECT_ROOT / "src" / "gi_vqa" / "contract.py"
    installed_contract_sha256 = hashlib.sha256(installed_contract_path.read_bytes()).hexdigest()
    source_contract_sha256 = hashlib.sha256(source_contract_path.read_bytes()).hexdigest()
    if installed_contract_sha256 != source_contract_sha256:
        raise RuntimeError("Installed contract runner differs from the verified checkout")

    git_status = run_command(
        ["git", "status", "--porcelain=v1", "--untracked-files=all"],
        cwd=CHECKOUT_ROOT,
        capture=True,
    ).stdout.strip()
    if git_status:
        raise RuntimeError(f"Verified checkout became dirty: {git_status.splitlines()}")

    bootstrap_record = {
        "status": "PASS",
        "python": sys.version,
        "implementation": platform.python_implementation(),
        "gpu_query": gpu_query,
        "repository_url": REPOSITORY_URL,
        "repository_commit": resolved_commit,
        "checkout_clean": True,
        "installed_contract_path": str(installed_contract_path),
        "installed_contract_sha256": installed_contract_sha256,
        "global_pip_check_exit_code": pip_check.returncode,
        "scoped_pip_conflicts": scoped_pip_conflicts,
    }
    (ARTIFACT_DIR / "bootstrap_environment.json").write_text(
        json.dumps(bootstrap_record, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    BOOTSTRAP_OK = True
    print("Bootstrap PASS: exact clean checkout and dependency installation are ready.")
except BaseException as exc:
    failure_message = re.sub(
        r"\b(?:hf|api)_[A-Za-z0-9_-]{12,}\b", "<redacted-token>", str(exc)
    )
    failure_traceback = re.sub(
        r"\b(?:hf|api)_[A-Za-z0-9_-]{12,}\b",
        "<redacted-token>",
        traceback.format_exc(),
    )
    failure = {
        "status": "FAIL",
        "phase": "bootstrap",
        "type": f"{type(exc).__module__}.{type(exc).__name__}",
        "message": failure_message,
        "traceback": failure_traceback,
    }
    (ARTIFACT_DIR / "bootstrap_failure.json").write_text(
        json.dumps(failure, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    print(f"Bootstrap FAIL: {failure_message}")

In [ ]:
import os

os.environ["HF_HOME"] = "/content/huggingface-cache"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_MODE"] = "disabled"

AUTH_OK = False
HF_USERNAME = None
if not BOOTSTRAP_OK:
    print("Authentication SKIPPED because bootstrap failed; evidence will still be bundled.")
else:
    try:
        from google.colab import userdata
        from huggingface_hub import login, whoami

        hf_token = userdata.get("HF_TOKEN")
        if not isinstance(hf_token, str) or not hf_token.strip():
            raise RuntimeError("Colab secret HF_TOKEN is missing or notebook access is disabled")
        login(token=hf_token, add_to_git_credential=False)
        hf_identity = whoami(token=hf_token)
        HF_USERNAME = hf_identity.get("name") or hf_identity.get("fullname")
        del hf_token
        AUTH_OK = True
        print(f"Hugging Face authentication PASS for user: {HF_USERNAME}")
    except BaseException as exc:
        auth_message = str(exc)
        try:
            if isinstance(hf_token, str):
                auth_message = auth_message.replace(hf_token, "<redacted-token>")
        except NameError:
            pass
        auth_message = re.sub(
            r"\b(?:hf|api)_[A-Za-z0-9_-]{12,}\b", "<redacted-token>", auth_message
        )
        failure = {
            "status": "FAIL",
            "phase": "authentication",
            "type": f"{type(exc).__module__}.{type(exc).__name__}",
            "message": auth_message,
        }
        (ARTIFACT_DIR / "authentication_failure.json").write_text(
            json.dumps(failure, indent=2, sort_keys=True) + "\n",
            encoding="utf-8",
        )
        try:
            del hf_token
        except NameError:
            pass
        print(f"Authentication FAIL: {auth_message}")

In [ ]:
contract_command = [
    sys.executable,
    "-m",
    "gi_vqa.contract",
    "--config",
    str(CONFIG_PATH),
    "--repository-root",
    str(CHECKOUT_ROOT),
    "--expected-commit",
    REPOSITORY_COMMIT,
    "--artifact-dir",
    str(ARTIFACT_DIR),
    "--required-gpu-substring",
    "T4",
]
if BOOTSTRAP_OK and AUTH_OK:
    CONTRACT_PROCESS = subprocess.run(contract_command, check=False)
else:
    CONTRACT_PROCESS = subprocess.CompletedProcess(contract_command, returncode=1)
    failed_phase = "bootstrap" if not BOOTSTRAP_OK else "authentication"
    source_failure = ARTIFACT_DIR / f"{failed_phase}_failure.json"
    early_message = f"Contract skipped because {failed_phase} failed"
    if source_failure.is_file():
        early_failure = json.loads(source_failure.read_text(encoding="utf-8"))
        early_message = early_failure.get("message", early_message)
    (ARTIFACT_DIR / "contract_report.json").write_text(
        json.dumps(
            {
                "status": "FAIL",
                "phase": failed_phase,
                "checks": {},
                "artifact_paths": {
                    "report": str(ARTIFACT_DIR / "contract_report.json"),
                },
                "error": {
                    "type": "EarlyContractFailure",
                    "message": early_message,
                },
            },
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )
REPORT_PATH = ARTIFACT_DIR / "contract_report.json"
if REPORT_PATH.is_file():
    CONTRACT_REPORT = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
else:
    CONTRACT_REPORT = {
        "status": "FAIL",
        "error": {
            "type": "ContractProcessFailure",
            "message": "The contract process ended without writing its required report",
        },
    }
    REPORT_PATH.write_text(
        json.dumps(CONTRACT_REPORT, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
print(f"Contract process exit code: {CONTRACT_PROCESS.returncode}")
print(f"Contract status: {CONTRACT_REPORT['status']}")
print(f"Machine-readable report: {REPORT_PATH}")

In [ ]:
from IPython.display import Markdown, display

checks = CONTRACT_REPORT.get("checks", {})
passed_checks = sum(bool(value.get("passed")) for value in checks.values())
display(
    Markdown(
        f"## Contract {CONTRACT_REPORT['status']}  \n"
        f"Recorded checks passed: **{passed_checks}/{len(checks)}** "
        "(later stages do not run after the first failure)"
    )
)

if CONTRACT_REPORT.get("status") == "PASS":
    generation = CONTRACT_REPORT["generation"]
    score = CONTRACT_REPORT["generation_score_verification"]
    item = CONTRACT_REPORT["diagnostic_item"]
    print(f"Item ID: {item['item_id']} (contract-only)")
    print(f"Question: {item['question']}")
    print(f"Generated answer: {generation['display_text']}")
    print(
        "Generation/teacher-forcing maximum |Δ log p|: "
        f"{score['maximum_absolute_difference']:.8g}"
    )
    swift_equivalence = CONTRACT_REPORT["swift_template_equivalence"]
    correction = swift_equivalence["builtin_token_type_correction"]
    print(
        "Built-in ms-swift boundary observation: "
        f"{correction['action']} at indices {correction['mismatch_indices']}"
    )
    print(
        "Validated Study 1 training template: "
        f"{swift_equivalence['study1_template_type']}"
    )

    maps_path = Path(CONTRACT_REPORT["artifact_paths"]["attribution_maps"])
    if maps_path.is_file():
        import matplotlib.pyplot as plt
        import numpy as np

        with np.load(maps_path) as maps:
            names = list(maps.files)
            figure, axes = plt.subplots(1, len(names), figsize=(5 * len(names), 4))
            axes = np.atleast_1d(axes)
            for axis, name in zip(axes, names):
                image = axis.imshow(maps[name], cmap="magma", vmin=0.0, vmax=1.0)
                axis.set_title(name.replace("_", " "))
                axis.axis("off")
                figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
            figure.tight_layout()
            plt.show()
else:
    error = CONTRACT_REPORT.get("error", {})
    print(f"Failure type: {error.get('type', 'unknown')}")
    print(f"Failure message: {error.get('message', 'No message recorded')}")
    failed_names = [name for name, value in checks.items() if not value.get("passed")]
    print(f"Failed checks: {failed_names}")

In [ ]:
import hashlib
import zipfile

pip_freeze = subprocess.run(
    [sys.executable, "-m", "pip", "freeze", "--all"],
    check=True,
    capture_output=True,
    text=True,
).stdout
(ARTIFACT_DIR / "pip_freeze.txt").write_text(pip_freeze, encoding="utf-8")

bundle_members = [
    path
    for path in sorted(ARTIFACT_DIR.iterdir())
    if path.is_file() and path.name != "contract_bundle.zip"
]
member_hashes = {
    path.name: hashlib.sha256(path.read_bytes()).hexdigest()
    for path in bundle_members
}
bundle_manifest = {
    "contract_status": CONTRACT_REPORT["status"],
    "repository_commit": REPOSITORY_COMMIT,
    "members_sha256": member_hashes,
}
manifest_path = ARTIFACT_DIR / "bundle_manifest.json"
manifest_path.write_text(
    json.dumps(bundle_manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
bundle_members.append(manifest_path)

BUNDLE_PATH = ARTIFACT_DIR / "contract_bundle.zip"
with zipfile.ZipFile(BUNDLE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in bundle_members:
        archive.write(path, arcname=path.name)
BUNDLE_SHA256 = hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest()
print(f"Evidence bundle: {BUNDLE_PATH}")
print(f"Bundle SHA-256: {BUNDLE_SHA256}")

if DOWNLOAD_BUNDLE:
    from google.colab import files

    files.download(str(BUNDLE_PATH))

In [ ]:
if CONTRACT_PROCESS.returncode != 0 or CONTRACT_REPORT.get("status") != "PASS":
    raise RuntimeError(
        "Backend contract FAILED. Keep the downloaded evidence bundle and fix the recorded "
        "failure before attempting the 20-item smoke run or any paper experiment."
    )
print(
    "Backend contract PASS. Preserve the evidence bundle with this Git commit. "
    "Next gate: create/audit the grouped split manifest, then implement the restart-safe "
    "20-item end-to-end smoke run."
)